# Notebook 2: Failure Injection & Degradation Dynamics

**GPU Fleet Autopilot — Research & Simulation Suite**

This notebook analyzes the **10 failure injection scenarios** across the simulated GPU fleet, examining their progressive degradation trajectories and ground-truth horizon labels ($2\text{h}$, $6\text{h}$, $24\text{h}$).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 6)

df = pd.read_csv("../data/sample_telemetry.csv")
print(f"Loaded dataset with {len(df)} records across {df['gpu_id'].nunique()} GPUs.")
print("Failure types distribution:")
print(df["failure_type"].value_counts())

## 1. Comparing Degradation Curves Across Scenarios

We inspect how each failure scenario affects temperature, compute throughput, and error counters.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Thermal failure: temperature creep
thermal_gpus = df[df["failure_type"] == "GPU_THERMAL_FAILURE"]["gpu_id"].unique()
if len(thermal_gpus) > 0:
    t_gpu = df[df["gpu_id"] == thermal_gpus[0]]
    axes[0, 0].plot(t_gpu["dcgm_gpu_temp"], color="crimson", label="Die Temp (°C)")
    axes[0, 0].axhline(92, color="darkred", linestyle="--", label="Throttle (92°C)")
    axes[0, 0].set_title(f"GPU_THERMAL_FAILURE ({thermal_gpus[0]})")
    axes[0, 0].legend()

# 2. ECC failure: SBE accumulation
ecc_gpus = df[df["failure_type"] == "GPU_ECC_FAILURE"]["gpu_id"].unique()
if len(ecc_gpus) > 0:
    e_gpu = df[df["gpu_id"] == ecc_gpus[0]]
    axes[0, 1].plot(e_gpu["dcgm_ecc_sbe_volatile_total"], color="purple", label="SBE Cumulative")
    axes[0, 1].set_title(f"GPU_ECC_FAILURE ({ecc_gpus[0]})")
    axes[0, 1].legend()

# 3. Straggler / Performance Regression
strag_gpus = df[df["failure_type"] == "PERFORMANCE_REGRESSION"]["gpu_id"].unique()
if len(strag_gpus) > 0:
    s_gpu = df[df["gpu_id"] == strag_gpus[0]]
    axes[1, 0].plot(s_gpu["performance_ratio"], color="orange", label="Performance Ratio")
    axes[1, 0].set_title(f"PERFORMANCE_REGRESSION ({strag_gpus[0]})")
    axes[1, 0].legend()

# 4. NVLink degradation
nv_gpus = df[df["failure_type"] == "NVLINK_DEGRADATION"]["gpu_id"].unique()
if len(nv_gpus) > 0:
    n_gpu = df[df["gpu_id"] == nv_gpus[0]]
    axes[1, 1].plot(n_gpu["dcgm_nvlink_error_count"], color="teal", label="NVLink Replays")
    axes[1, 1].set_title(f"NVLINK_DEGRADATION ({nv_gpus[0]})")
    axes[1, 1].legend()

plt.tight_layout()
plt.show()